In [7]:
import os
import torch
from safetensors.torch import load_file
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification

# ============================================================
# CONFIG
# ============================================================

BASE_MODEL = "indolem/indobertweet-base-uncased"
MODEL_ID = "komangytri/indobertweet_metadate_v1"

LABEL_COLS = [
    "insults",
    "identity_attack",
    "threat_incitement_to_violence",
    "profanity_obscenity",
    "sexually_explicit"
]

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

Device: cpu


In [8]:
# ============================================================
# LOAD TOKENIZER
# ============================================================

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)


tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL
)



In [9]:
# ============================================================
# LOAD MODEL DARI HUGGING FACE
# ============================================================

model_2 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID
)

print("Model berhasil dimuat dari Hugging Face.")


# ============================================================
# DEVICE + EVAL
# ============================================================

model_2.to(device)
model_2.eval()

print("Model siap untuk prediction.")

config.json:   0%|          | 0.00/1.14k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  442MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model berhasil dimuat dari Hugging Face.
Model siap untuk prediction.


In [10]:
import numpy as np
import torch


# LABEL_COLS = [
#     "insults",
#     "identity_attack",
#     "threat_incitement_to_violence",
#     "profanity_obscenity",
#     "sexually_explicit"
# ]


# device = torch.device(
#     "cuda" if torch.cuda.is_available() else "cpu"
# )

# model_2.to(device)
# model_2.eval()


def predict_multilabel(
    text,
    ethnicity,
    religion,
    disability,
    lgbt,
    gender,
    age,
    domisili,
    pendidikan,
    pekerjaan,
    president_vote
):

    # ========================================================
    # METADATA ANNOTATOR
    # ========================================================

    metadata_text = (
        f"Annotator memiliki etnis {ethnicity}, "
        f"agama {religion}, "
        f"status disabilitas {disability}, "
        f"status LGBT {lgbt}, "
        f"gender {gender}, "
        f"usia {age}, "
        f"berdomisili di {domisili}, "
        f"pendidikan terakhir {pendidikan}, "
        f"status pekerjaan {pekerjaan}, "
        f"dan kecenderungan pilihan presiden {president_vote}."
    )


    # ========================================================
    # COMBINE TEXT + METADATA
    # ========================================================

    combined_text = (
        f"{metadata_text} "
        f"Komentar yang dianotasi: {text}"
    )


    # ========================================================
    # TOKENIZATION
    # ========================================================

    inputs = tokenizer(
        combined_text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )


    # ========================================================
    # MOVE INPUT TO DEVICE
    # ========================================================

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }


    # ========================================================
    # PREDICTION
    # ========================================================

    with torch.no_grad():

        outputs = model_2(**inputs)

        logits = outputs.logits

        probs = torch.sigmoid(logits)

        probs = probs.cpu().numpy()[0]


    # ========================================================
    # THRESHOLD
    # ========================================================

    predictions = (
        probs >= 0.5
    ).astype(int)


    # ========================================================
    # RESULT
    # ========================================================

    result = {}

    for label, pred, prob in zip(
        LABEL_COLS,
        predictions,
        probs
    ):

        result[label] = {
            "prediction": int(pred),
            "probability": float(prob)
        }


    return {
        "combined_text": combined_text,
        "results": result
    }

In [11]:
result = predict_multilabel(
    text="Israel terus menyerang Palestina, pemerintah dunia cuma diam aja",

    ethnicity="Jawa",
    religion="Islam",
    disability="Tidak",
    lgbt="Tidak",
    gender="Perempuan",
    age=21,
    domisili="Jakarta",
    pendidikan="S1",
    pekerjaan="Mahasiswa",
    president_vote="Prabowo"
)

result

{'combined_text': 'Annotator memiliki etnis Jawa, agama Islam, status disabilitas Tidak, status LGBT Tidak, gender Perempuan, usia 21, berdomisili di Jakarta, pendidikan terakhir S1, status pekerjaan Mahasiswa, dan kecenderungan pilihan presiden Prabowo. Komentar yang dianotasi: Israel terus menyerang Palestina, pemerintah dunia cuma diam aja',
 'results': {'insults': {'prediction': 1, 'probability': 0.7062894105911255},
  'identity_attack': {'prediction': 0, 'probability': 0.03855874016880989},
  'threat_incitement_to_violence': {'prediction': 0,
   'probability': 0.26352599263191223},
  'profanity_obscenity': {'prediction': 0, 'probability': 0.09253216534852982},
  'sexually_explicit': {'prediction': 0, 'probability': 0.01343456655740738}}}